# Kinematics |  Angular Head Velocity
-----------------------------------------

## Setup and Configuration

In [ ]:
##### Imports ##########################################################################

#==== Generic Imports ================
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import pandas as pd
from IPython.display import display, Markdown
from collections.abc import Callable

#==== Movement Imports ===============
import movement.kinematics
import movement.utils.vector

#==== DataConduit Imports (Core) =====
import data_conduit as dc
from dc.datastructures import slice_stream, slice_stream_per_trial
from dc.integrations.DLC.pose import pose_to_movement  # Import used for combining DLC pose & confidence streams into a single object
#       matches `movement`'s own conventions

#==== DataConduit Imports (Unique) ===
from movement_figures.data_template.loading import build_datastructure

from movement_figures.misc import visual_streams_check, configure_simplified_trial_df
from movement_figures.timeseries_template.annotations import annotate_qc_timeseries

from data_conduit.refactor_qc import (  # Imports for trial filtering based on experiment protocol
    TRAINING_FIRST_MID_LAST_DETAIL,
    training_spec,
    training_session_names,
    filter_trials,
)
from movement_figures.timeseries_template.annotations import AnnotationResult

### Notebook Settings ######################################################

pd.set_option("display.max_columns", None)  # Changes default pandas display settings to show all columns in a dataframe.
pd.set_option("display.max_rows", None)  # Changes default pandas display settings to show all rows in a dataframe.

In [ ]:
##### Setup Parameters #################################################################

# === 1| Select Mice, Days and Recording Folders Before Reading Files =============
ROOT = Path("/media/sepi/Elements/PathIntegrationProtocol/BonsaiOutput/Training")

LEVEL_SELECTORS = {
    'l0_selector': 'MR_M01569515', 
    'l1_selector': 'Day21'
    }                                                                                       # Example choices belong here: l0_selector and l1_selector.
# INCLUDE_SESSIONS = ('2026-06-21T093349Z',)                                                # Optional list of recording folder names; None keeps selected folders.
# STREAMS = ('trials', 'events', 'video', 'dlc')                                            # These figures require events/trials and aligned pose; other sources remain available.


#=== 2| Set the trial filtering parameters for the experiment protocol ============
spec = training_spec(TRAINING_FIRST_MID_LAST_DETAIL)                                        # Flattens `{training_detail}` table into per-session spec. 

# included_sessions = training_session_names(spec)                                          # Returns a list of session folder names that match the spec.
#   Not used here, as sessions have been filtered in another way. If other sessions 
#   not in the above spec are used, then applying the `filter_trials` function with 
#   the above spec could result in incorrect filtering.


In [ ]:
##### Load DataStructure ###############################################################

#=== Load data ========================================================================

fig_ds_obj = build_datastructure(
    root               = ROOT,                                                                 # Directory above the named hierarchy levels.
    level_names        = ("mouseID", "day"),                                                   # Lab layout: root / mouseID / day / session.
    streams            = ("trials", "session_settings", "video", "dlc"),                       # Data streams to be included in the datastructure.
    include            = None,                                                                 # Keep these session folder names; None keeps all.
    exclude            = None,                                                                 # Alternatively omit these session folder names.
    raw_events         = True,                                                                 # Convenience alias for adding "events" to the requested sources.
    device_yaml        = "./device.yml",                                                       # Lab Nosepoke board register definitions.
    soundcard_yaml     = "./soundcard.yml",                                                    # Lab SoundCard register definitions.
    nosepoke_count     = 18,                                                                   # Port count used by the existing Q_C parser.
    trial_start_buffer = 0.0,                                                                  # Preserve the existing lab parser setting.
    dlc_file_format     = "auto",                                                              # Existing reader prefers CSV, otherwise HDF5.
    trial_spec = None,                                                                         # Optional experiment rules; None keeps the existing Q_C specification. 
    #    Not used here, as sessions are already selected. 
    #    If session is not one in the spec, then subsequent use of 
    #   `filter_trials` to apply correct experiment protocol rules could be incorrect.
    
    **LEVEL_SELECTORS,
)


#=== Select and load the streams of interest from datastructure. ======================
fig_ds_obj.select()
streams = fig_ds_obj.load()                                                                    # Load is required to access the selected streams.

#=== Create convenient compact trial dataframe without some unnecessary columns. ======
streams['compact_trials'] = configure_simplified_trial_df(              
    trial_df = streams['trials'],
    modify_in_place = False,                                                                   # Whether to modify the trial dataframe in place or return a new dataframe. 
)
'''More useful for visual inspection and analysis. 
Check the misc.py file for the default columns to keep and drop.'''


#=== Apply Trial Filtering Protocol to construct new stream objects ===================

# Full filtered trials
streams['filtered_trials'] = filter_trials(
    trials = streams['trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

# Filtered compact trials 
streams['filtered_compact_trials'] = filter_trials(
    trials = streams['compact_trials'],
    spec = spec,
    # session_column = 'session', mouse_column = 'mouseID', trial_index_column = 'trial_index', led_column = 'LED', report = True,
    #                                                            # Leave params untouched for now, let the function use its defaults.
)

#=== Merge Pose & Confidence Streams ==================================================
streams['dlc:movement'] = pose_to_movement(
    slice_stream(streams["dlc:position"],   
                #selectors={"session": session_id}
    ),
    slice_stream(streams["dlc:confidence"], 
                 #selectors={"session": session_id}
    ),
)

#=== Visual streams check =============================================================
visual_streams_check(
    loaded_ds_object = streams,                                                                # Loaded dataset object containing all streams of interest.
    show_stream_shapes = True,                                                                 # Whether to display the shapes of the streams.
    show_stream_dtypes = True,                                                                 # Whether to display the data types of the streams.
    display_streams = False                                                                    # Whether to display the actual stream data as a table.
)



## Select Trials to Show

In [ ]:
##### Trial Selection ##################################################################

#==== Extract Trial Range ============
selected_trials = slice_stream(streams['filtered_trials'][:8])              # Select the first n trials

#==== Extract Recording Identifier ===
sessionID = selected_trials['session'].unique().item()                      # Extract unique sID from selected trials, remove duplicates, ensure single value.

#==== Select Tracked Positions =======
recording = slice_stream(stream = streams['dlc:movement'],                  # selectors{} here keeps movement data belonging to the specified session only.
                        selectors = {'session' : sessionID }, 
)

position = recording['position']                                            # ['position'] extracts positional data from the movement stream.

position = position.sel(individual='individual_0',                          # .sel() selects the specified individual from the positional data.
                        drop=True,                                          # drop=True removes the individual dimension from the resulting DataArray, but leaves other dims intact.
)


## Compute Angular Head Velocity

In [ ]:
##### Compute Angular Head Velocity Using `movement` ###################################

#==== Calculate Head Direction =======


head_direction = movement.kinematics.compute_head_direction_vector(
    data=position,
    left_keypoint="lear",
    right_keypoint="rear",
    camera_view="top_down",
)


#==== Calculate Change in Head Dir. ==

previous_head_direction = head_direction.shift(time=1)              # Shift head direction by one time step to get the previous frame.

angular_change = movement.utils.vector.compute_signed_angle_2d(
    u=previous_head_direction,                                      # Head direction at the previous frame.
    v=head_direction,                                               # Head direction at the current frame.
)


#==== Convert Angular Change to Vel. =
elapsed_time = (head_direction.time - head_direction.time.shift(time=1))

angular_head_velocity = angular_change / elapsed_time
angular_head_velocity.name = "angular_head_velocity"
angular_head_velocity.attrs["units"] = "rad/s"


#==== Slice AHV to Selected Range ====

selected_angular_head_velocity = slice_stream_per_trial(
    stream=angular_head_velocity,
    trials=selected_trials,
    segment="trial",
    time_coord="time",
)

selected_angular_head_velocity = xr.concat(
    selected_angular_head_velocity,
    dim="time",
)  

## Angular Head Velocity Plotting Function

In [ ]:
def add_annotations(ax, 
                    selected_trials,
                    **kwargs
                    ):
    '''
    Add annotations to the given axis using the selected trials.
    
    Parameters
    ----------
    ax : matplotlib.axes.Axes
        The axis to which annotations will be added.
    selected_trials : pd.DataFrame
        The selected trials used for annotations.

    Returns
    -------
    matplotlib.axes.Axes
        The axis with annotations added.
    '''
    return annotate_qc_timeseries(ax=ax, trials=selected_trials, **kwargs)

In [ ]:
##### Plot Angular Head Velocity on Ax #################################################


def plot_ax_angular_head_velocity(
    ax: plt.Axes,
    angular_head_velocity: xr.DataArray,
    selected_trials: pd.DataFrame,
    units: str = "deg/s",
    row_duration: float | None = 60.0,
    subtitle: str | None = "Angular Head Velocity",
    add_annotations_func: Callable[..., AnnotationResult] | None = add_annotations,
    annotation_kwargs: dict | None = None,
    **kwargs,
) -> plt.Axes:
    """
    Plot angular head velocity in consecutive time rows.

    Parameters
    ----------
    ax : plt.Axes
        Axis used for the angular-head-velocity plot.
    angular_head_velocity : xr.DataArray
        Signed angular head velocity in radians per second.
    units : str
        Output units, either ``"deg/s"`` or ``"rad/s"``.
    row_duration : float or None
        Duration of each plotted row in seconds. None plots one row.
    subtitle : str or None
        Plot title.
    add_annotations_func : callable or None
        Optional function for adding annotations to each row.
    annotation_kwargs : dict or None
        Keyword arguments passed to the annotation function.
    **kwargs
        Additional arguments passed to ``Axes.plot``.

    Returns
    -------
    plt.Axes
        Axis containing the angular-head-velocity plot.
    """

    #==== 0| Validation Checks =======================================

    if ax is None:
        raise ValueError("ax cannot be None")

    if angular_head_velocity is None:
        raise ValueError("angular_head_velocity cannot be None")


    #==== 1| Calculate Rows Needed for Subplot =======================

    range_start = float(angular_head_velocity['time'][0])
    range_end   = float(angular_head_velocity['time'][-1])

    duration = range_end - range_start if row_duration is None else row_duration

    row_count = max(1, int(np.ceil((range_end - range_start) / duration)))


    #==== 2| Create the Required Time Rows ===========================
    if row_count == 1:
        row_axes = [ax]

    else:
        ax.set_axis_off()                                                           # Use the supplied axis as the container for the rows.

        row_axes = []                                                               # List to hold the axes for each row
        row_height = 1 / (row_count + 0.25 * (row_count - 1))                       # Calculate the height of each row considering the gaps between them.
        row_gap = 0.25 * row_height                                                 # Calculate the gap between rows as a fraction of the row height.

        for row in range(row_count):                                                # Create an inset axis for each row
            bottom = 1 - (row + 1) * row_height - row * row_gap                     # Calculate the bottom position of the current row.
            row_axes.append(                                                        # Append the inset axis for the current row to the list of row axes.
                ax.inset_axes([0, bottom, 1, row_height])
            )


    #==== 3| Convert Units if Necessary ==============================

    if units == "deg/s":
        adjusted_ahv = np.rad2deg(angular_head_velocity)

    elif units == "rad/s":
        adjusted_ahv = angular_head_velocity

    else:
        raise ValueError("units must be 'deg/s' or 'rad/s'.")


    #==== 4| Set Plotting Parameters =================================

    y_limit = float(
        np.nanmax(np.abs(adjusted_ahv.values))
    )



    #==== 5| Plot AHV and Matching Annotations =======================

    annotation_kwargs = annotation_kwargs or {'selected_trials': selected_trials} if add_annotations_func is add_annotations else {}

    for row, row_ax in enumerate(row_axes):

        row_start = range_start + row * duration                        # Calculate the start time for the current row
        row_end = min(row_start + duration, range_end)                  # Calculate the end time for the current row

        row_ahv = adjusted_ahv.sel(time=slice(row_start, row_end))      # Select the portion of angular head velocity data for the current row

        row_ax.plot(row_ahv.time,                                       # Plot the angular head velocity for the current row
                    row_ahv,
                    ".",
                    ms=2.5,
                    **kwargs,
        )

        row_ax.axhline(0, linewidth=0.8)                               # Add a horizontal line at y=0 for reference

        row_ax.set_xlim(row_start, row_start + duration)               # Set the x-axis limits for the current row
        row_ax.set_ylim(-y_limit, y_limit,)                            # Set the y-axis limits for the current row
    
        row_ax.set_ylabel(f"Angular Head Velocity ({units})")          # Set the y-axis label for the current row
        row_ax.set_xlabel("Time (Seconds)")                            # Set the x-axis label for the current row


        # Keep annotation coordinates in the original recording-time units.
        if add_annotations_func is not None:
            if selected_trials is None:
                raise ValueError("selected_trials must be provided")

            add_annotations_func(
                row_ax,
                selected_trials,
                **(annotation_kwargs or {}),
            )

    #==== 6| Add the Plot Title ======================================

    if subtitle is not None:
        row_axes[0].set_title(subtitle)

    return ax

## Generate Plot

In [ ]:
fig, ax = plt.subplots()

ax = plot_ax_angular_head_velocity(ax = ax,
                                   selected_trials=selected_trials,
                                   units = "deg/s",
                                   row_duration=60.0,
                                   subtitle="Angular Head Velocity",
                                   add_annotations_func=add_annotations,
                                   annotation_kwargs=None,
)

plt.show()
